# Ручной бейзлайн

## Настройки

In [1]:
%pip -q install "catboost>=1.2" "pyarrow>=16" "huggingface_hub>=0.30" psutil tqdm

In [2]:
import gc
import json
import hashlib
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, log_loss
from huggingface_hub import hf_hub_download
from google.colab import drive

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/TLab_Yambda/retrieval_gts_v2')
OUT = Path('/content/drive/MyDrive/TLab_Yambda/catboost_reranking_v1')
FEAT = OUT / 'features'
MODELS = OUT / 'models'
for folder in (OUT, FEAT, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 666
DAY = 86400
WINDOWS = (1, 7, 30)
KS = (5, 10, 20, 100)
ITERATIONS = 800
DEPTH = 7
LEARNING_RATE = 0.05

snapshots = pd.read_csv(DATA / 'snapshots.csv').set_index('split')
config = json.loads((DATA / 'config.json').read_text())
print('Retrieval:', config.get('n_users'), 'users,', config.get('n_candidates'), 'candidates/user')
display(snapshots)
print('Saved results:', OUT)

import subprocess, sys
with (OUT/'requirements-lock.txt').open('w') as lockfile:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=lockfile, check=True)


Mounted at /content/drive
Retrieval: 10000 users, 600 candidates/user


,snapshot_id,cutoff,history_end,end
split,,,,
train,0,25737200,25735400,25823600
valid,1,25825400,25823600,25911800
test,2,25913600,25911800,26000000


Saved results: /content/drive/MyDrive/TLab_Yambda/catboost_reranking_v1


## История

In [3]:
columns = ['uid', 'item_id', 'timestamp', 'event_type', 'played_ratio_pct',
           'is_organic', 'track_length_seconds', 'positive']
events = pd.read_parquet(DATA / 'events_sample.parquet', columns=columns)
events['event_type'] = events.event_type.astype('category')
events['played_ratio_pct'] = events.played_ratio_pct.astype('float32')
events['track_length_seconds'] = events.track_length_seconds.astype('float32')
if not events.timestamp.is_monotonic_increasing:
    events = events.sort_values('timestamp', kind='stable').reset_index(drop=True)
event_times = events.timestamp.to_numpy(dtype=np.int64)

mapping_file = hf_hub_download(repo_id='yandex/yambda', repo_type='dataset',
                               filename='artist_item_mapping.parquet')
artist_map = pd.read_parquet(mapping_file)
print('Mapping columns:', artist_map.columns.tolist())
artist_map = artist_map[['item_id', 'artist_id']].dropna().drop_duplicates()
artist_map = artist_map.astype({'item_id': 'uint32', 'artist_id': 'uint32'})
print(f'Events: {len(events):,} · Item–artist links: {len(artist_map):,}')


artist_item_mapping.parquet: reconstructing file:   0%|          |  0.00B / 38.3MB            

artist_item_mapping.parquet: downloading bytes:           |  0.00B            

Mapping columns: ['artist_id', 'item_id']
Events: 47,790,449 · Item–artist links: 9,271,906


## Агрегаты

In [4]:
def entity_features(past, history_end, key, prefix):
    listens = past.loc[past.event_type.eq('listen')]
    all_g = listens.groupby(key, sort=False, observed=True)
    all_stats = all_g.agg(n=('item_id', 'size'), pos=('positive', 'sum'),
                           last=('timestamp', 'max'))
    all_stats = all_stats.rename(columns={'n': f'{prefix}_listens_all',
                                           'pos': f'{prefix}_pos_all',
                                           'last': f'{prefix}_last_ts'})
    other_key = 'item_id' if key == 'uid' else 'uid'
    for days in WINDOWS:
        w = past.loc[past.timestamp.ge(history_end - days * DAY)]
        l = w.loc[w.event_type.eq('listen')]
        g = l.groupby(key, sort=False, observed=True)
        a = g.agg(n=('item_id', 'size'), pos=('positive', 'sum'),
                  unique=(other_key, 'nunique'))
        a.columns = [f'{prefix}_{name}_{days}d' for name in a.columns]
        all_stats = all_stats.join(a, how='outer')
        if days == 30:
            extra = g.agg(ratio=('played_ratio_pct', 'mean'),
                          organic=('is_organic', 'mean'),
                          length=('track_length_seconds', 'mean'))
            extra.columns = [f'{prefix}_{name}_30d' for name in extra.columns]
            all_stats = all_stats.join(extra, how='outer')
            negative = w.loc[w.event_type.eq('dislike')].groupby(key, observed=True).size()
            all_stats[f'{prefix}_dislikes_30d'] = negative
    all_stats[f'{prefix}_last_age_days'] = (history_end - all_stats.pop(f'{prefix}_last_ts')) / DAY
    for span in (7, 30):
        all_stats[f'{prefix}_positive_rate_{span}d'] = (
            (all_stats[f'{prefix}_pos_{span}d'].fillna(0) + 1) /
            (all_stats[f'{prefix}_n_{span}d'].fillna(0) + 2))
    if prefix == 'u':
        all_stats['u_repeat_ratio_30d'] = 1 - all_stats.u_unique_30d / all_stats.u_n_30d
        all_stats['u_repeat_ratio_30d'] = all_stats.u_repeat_ratio_30d.clip(0, 1)
    else:
        all_stats['i_popularity_trend'] = (
            all_stats.i_n_1d.fillna(0) /
            (all_stats.i_n_7d.fillna(0) / 7 + 1))
    return all_stats.astype('float32')


def pair_features(past, candidate_pairs, history_end):

    cand_key = ((candidate_pairs.uid.to_numpy(dtype=np.uint64) << np.uint64(32)) |
                candidate_pairs.item_id.to_numpy(dtype=np.uint64))
    hist_key = ((past.uid.to_numpy(dtype=np.uint64) << np.uint64(32)) |
                past.item_id.to_numpy(dtype=np.uint64))
    mask = np.isin(hist_key, np.unique(cand_key))
    del hist_key
    h = past.loc[mask, ['uid','item_id','timestamp','event_type',
                        'positive','played_ratio_pct','is_organic']].copy()
    h['pair_key'] = ((h.uid.to_numpy(dtype=np.uint64) << np.uint64(32)) |
                     h.item_id.to_numpy(dtype=np.uint64))
    g = h.groupby('pair_key', sort=False, observed=True)
    result = g.agg(ui_events_all=('item_id','size'), ui_last_ts=('timestamp','max'))
    listens = h.loc[h.event_type.eq('listen')]
    lg = listens.groupby('pair_key', sort=False, observed=True)
    stats = lg.agg(ui_listens_all=('item_id','size'), ui_positives_all=('positive','sum'),
                   ui_first_ts=('timestamp','min'), ui_last_listen_ts=('timestamp','max'),
                   ui_mean_played_ratio=('played_ratio_pct','mean'),
                   ui_organic_share=('is_organic','mean'))
    result = result.join(stats, how='left')
    last_ratio = lg.played_ratio_pct.last()
    result['ui_last_played_ratio'] = last_ratio
    for days in (7, 30):
        w = listens.loc[listens.timestamp.ge(history_end - days * DAY)]
        agg = w.groupby('pair_key', sort=False, observed=True).agg(
            n=('item_id','size'), pos=('positive','sum'))
        result[f'ui_listens_{days}d'] = agg.n
        result[f'ui_positives_{days}d'] = agg.pos

    actions = h.loc[h.event_type.isin(['like','unlike','dislike','undislike']),
                    ['pair_key','event_type']]
    for yes, no, col in [('like','unlike','ui_like_active'),
                         ('dislike','undislike','ui_dislike_active')]:
        a = actions.loc[actions.event_type.isin([yes, no])]
        result[col] = a.groupby('pair_key', sort=False, observed=True).event_type.last().eq(yes)
    result['ui_last_age_days'] = (history_end - result.pop('ui_last_ts')) / DAY
    result['ui_last_listen_age_days'] = (history_end - result.pop('ui_last_listen_ts')) / DAY
    result['ui_first_listen_age_days'] = (history_end - result.pop('ui_first_ts')) / DAY
    return result.astype('float32')


In [5]:
def artist_features(base, past, history_end, mapping):
    recent = past.loc[past.timestamp.ge(history_end - 30 * DAY)]
    l = recent.loc[recent.event_type.eq('listen'),
                   ['uid','item_id','timestamp','positive']]
    joined = l.merge(mapping, on='item_id', how='inner', sort=False)
    artist_agg = joined.groupby(['uid','artist_id'], sort=False).agg(
        ua_listens_30d=('item_id','size'),
        ua_positives_30d=('positive','sum'),
        ua_unique_tracks_30d=('item_id','nunique'),
        ua_last_ts=('timestamp','max')).reset_index()
    artist_agg['ua_last_age_days'] = (history_end - artist_agg.pop('ua_last_ts')) / DAY
    artist_agg['ua_positive_rate_30d'] = (
        (artist_agg.ua_positives_30d + 1) / (artist_agg.ua_listens_30d + 2))
    del joined, l


    cand = base[['uid','item_id']].copy()
    cand['row_id'] = np.arange(len(cand), dtype=np.int32)
    cand = cand.merge(mapping, on='item_id', how='inner', sort=False)
    counts = cand.groupby('row_id', sort=False).size().rename('ua_artist_count')
    cand = cand.merge(artist_agg, on=['uid','artist_id'], how='left', sort=False)
    group = cand.groupby('row_id', sort=False)
    out = group.agg(
        ua_listens_30d_max=('ua_listens_30d','max'),
        ua_listens_30d_mean=('ua_listens_30d','mean'),
        ua_positives_30d_max=('ua_positives_30d','max'),
        ua_positives_30d_mean=('ua_positives_30d','mean'),
        ua_unique_tracks_30d_max=('ua_unique_tracks_30d','max'),
        ua_positive_rate_30d_max=('ua_positive_rate_30d','max'),
        ua_last_age_days_min=('ua_last_age_days','min'))
    out = out.join(counts, how='outer')

    return out.astype('float32')


## Признаки

In [6]:
CACHE_VERSION = 'manual_features_v2'

def candidate_fingerprint(frame):
    cols = ['uid', 'item_id', 'group_id', 'retrieval_rank', 'label',
            'itemknn_score', 'decaypop_score', 'retrieval_source']
    values = pd.util.hash_pandas_object(frame[cols], index=False).to_numpy(dtype=np.uint64)
    return hashlib.sha256(values.tobytes()).hexdigest()

def feature_manifest(split, candidates):
    return dict(version=CACHE_VERSION, split=split, seed=SEED,
                history_end=int(snapshots.loc[split].history_end),
                rows=len(candidates),
                candidates=candidate_fingerprint(candidates),
                history_bytes=(DATA / 'events_sample.parquet').stat().st_size,
                mapping_bytes=mapping_file.stat().st_size)

def feature_cache_valid(split):
    target = FEAT / f'features_{split}.parquet'
    manifest = FEAT / f'features_{split}.json'
    if not target.exists() or not manifest.exists():
        return False
    try:
        cols = ['uid', 'item_id', 'group_id', 'retrieval_rank', 'label',
                'itemknn_score', 'decaypop_score', 'retrieval_source']
        base = pd.read_parquet(DATA / 'candidates.parquet',
                               filters=[('split', '==', split)], columns=cols)
        base = base.sort_values(['group_id', 'retrieval_rank'],kind='stable').reset_index(drop=True)
        expected = feature_manifest(split, base)
        return (json.loads(manifest.read_text()) == expected
                and pq.ParquetFile(target).metadata.num_rows == len(base)
                and all(k in pq.read_schema(target).names for k in
                        ['uid','item_id','group_id','retrieval_rank','label',
                         'itemknn_score','decaypop_score','source_is_itemknn']))
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return False

def build_features(split):
    tic = time.perf_counter()
    snap = snapshots.loc[split]
    history_end = int(snap.history_end)
    start, end = int(snap.cutoff), int(snap.end)
    print(f'\n{split}: history < {history_end:,}; label window [{start:,}, {end:,})')

    base = pd.read_parquet(DATA / 'candidates.parquet', filters=[('split', '==', split)])
    base = base.sort_values(['group_id', 'retrieval_rank'], kind='stable').reset_index(drop=True)
    n_rows = len(base)
    manifest = feature_manifest(split, base)
    upper = np.searchsorted(event_times, history_end, side='left')
    past = events.iloc[:upper]
    pos_start = np.searchsorted(event_times, start, side='left')
    pos_end = np.searchsorted(event_times, end, side='left')
    future = events.iloc[pos_start:pos_end]
    future = future.loc[future.positive, ['uid', 'item_id']].drop_duplicates()
    truth_n = future.groupby('uid', observed=True).size()

    frame = base[['uid','item_id','group_id','retrieval_rank','label',
                  'itemknn_score','decaypop_score']].copy()
    frame['future_positive_total'] = base.uid.map(truth_n).fillna(0).astype('int32')
    frame['source_is_itemknn'] = base.retrieval_source.eq('itemknn').astype('int8')

    u = entity_features(past, history_end, 'uid', 'u')
    i = entity_features(past, history_end, 'item_id', 'i')
    frame = frame.join(u, on='uid').join(i, on='item_id')
    del u, i
    gc.collect()

    ui = pair_features(past, base[['uid','item_id']], history_end)
    frame['pair_key'] = ((frame.uid.to_numpy(dtype=np.uint64) << np.uint64(32)) |
                         frame.item_id.to_numpy(dtype=np.uint64))
    frame = frame.join(ui, on='pair_key').drop(columns='pair_key')
    del ui
    gc.collect()
    print('  User–item ->')

    ua = artist_features(base, past, history_end, artist_map)
    frame = frame.join(ua)
    del ua, base, future
    gc.collect()
    print('  User–artist ->')


    age_cols = [c for c in frame if '_age_days' in c]
    feature_cols = [c for c in frame if c.startswith(('u_', 'i_', 'ui_', 'ua_'))]
    counts = [c for c in feature_cols if c not in age_cols and not
              ('ratio' in c or 'rate' in c or 'share' in c or 'organic' in c or 'length' in c)]
    frame[counts] = frame[counts].fillna(0)

    frame[['ui_like_active','ui_dislike_active']] = frame[['ui_like_active','ui_dislike_active']].fillna(0)
    numeric = [c for c in frame.columns if c not in
               ['uid','item_id','group_id','label','future_positive_total']]
    frame[numeric] = frame[numeric].astype('float32')

    saved = FEAT / f'features_{split}.parquet'
    temporary = saved.with_suffix('.partial.parquet')
    frame.to_parquet(temporary, index=False, compression='zstd')
    os.replace(temporary, saved)
    (FEAT / f'features_{split}.json').write_text(json.dumps(manifest, indent=2))
    summary = {'split': split, 'pairs': n_rows, 'positive_candidates': int(frame.label.sum()),
               'users': int(frame.group_id.nunique()),
               'positive_groups': int(frame.groupby('group_id').label.max().sum()),
               'n_features': len(numeric), 'seconds': round(time.perf_counter()-tic, 1),
               'ram_gib': round(psutil.Process().memory_info().rss / 2**30, 2)}
    del frame
    gc.collect()
    print(' ', summary)
    return summary


In [7]:
feature_log = []
for split in tqdm(('train','valid','test'), desc='Feature engineering'):
    target = FEAT / f'features_{split}.parquet'
    if feature_cache_valid(split):
        print('Already on Drive:', target.name)
    else:
        feature_log.append(build_features(split))

if feature_log:
    pd.DataFrame(feature_log).to_csv(OUT / 'feature_time.csv', index=False)
print('Feature files:', [(p.name, round(p.stat().st_size / 2**20)) for p in FEAT.glob('*.parquet')])


Feature engineering:   0%|          | 0/3 [00:00<?, ?it/s]

Already on Drive: features_train.parquet
Already on Drive: features_valid.parquet
Already on Drive: features_test.parquet
Feature files: [('features_train.parquet', 226), ('features_valid.parquet', 227), ('features_test.parquet', 227)]


## Наборы признаков

In [8]:
schema = pq.read_schema(FEAT / 'features_train.parquet').names
RETRIEVAL_FEATURES = ['retrieval_rank','itemknn_score','decaypop_score','source_is_itemknn']
R1_FEATURES = RETRIEVAL_FEATURES + [c for c in schema if c.startswith(('u_', 'i_'))]
R2_FEATURES = R1_FEATURES + [c for c in schema if c.startswith(('ui_', 'ua_'))]

FEATURE_SETS = {'R1': R1_FEATURES, 'R2': R2_FEATURES}
(OUT / 'feature_sets.json').write_text(json.dumps(FEATURE_SETS, ensure_ascii=False, indent=2))
print('R1:', len(R1_FEATURES), 'features · R2:', len(R2_FEATURES), 'features')
print('R2-only:', [c for c in R2_FEATURES if c not in R1_FEATURES])


R1: 42 features · R2: 65 features
R2-only: ['ui_events_all', 'ui_listens_all', 'ui_positives_all', 'ui_mean_played_ratio', 'ui_organic_share', 'ui_last_played_ratio', 'ui_listens_7d', 'ui_positives_7d', 'ui_listens_30d', 'ui_positives_30d', 'ui_like_active', 'ui_dislike_active', 'ui_last_age_days', 'ui_last_listen_age_days', 'ui_first_listen_age_days', 'ua_listens_30d_max', 'ua_listens_30d_mean', 'ua_positives_30d_max', 'ua_positives_30d_mean', 'ua_unique_tracks_30d_max', 'ua_positive_rate_30d_max', 'ua_last_age_days_min', 'ua_artist_count']


## Метрики

In [9]:
def ranking_metrics(frame, scores, model_name):
    groups = int(frame.group_id.nunique())
    group_size = len(frame) // groups

    y = frame.label.to_numpy(dtype=np.int8).reshape(groups, group_size)
    truth = frame.future_positive_total.to_numpy(dtype=np.int32).reshape(groups, group_size)[:, 0]
    s = np.asarray(scores, dtype=np.float32).reshape(groups, group_size)
    order = np.argsort(-s, axis=1, kind='stable')[:, :max(KS)]
    ranked_y = np.take_along_axis(y, order, axis=1)
    available = y.sum(axis=1)
    weights = 1 / np.log2(np.arange(2, max(KS) + 2))
    dcg_cumsum = np.cumsum(ranked_y * weights, axis=1)
    idcg_cumsum = np.cumsum(weights)
    eligible = truth > 0
    found = available > 0
    result = {'model': model_name, 'split': '', 'users': groups,
              'positive_groups_pct': round(found.mean() * 100, 3),
              'eligible_groups_pct': round(eligible.mean() * 100, 3)}
    for k in KS:
        dcg = dcg_cumsum[:, k - 1]
        idcg_all = np.where(truth > 0, idcg_cumsum[np.maximum(np.minimum(truth, k)-1,0)], 1)
        idcg_found = np.where(found, idcg_cumsum[np.maximum(np.minimum(available, k)-1,0)], 1)
        ndcg_all = np.where(eligible, dcg / idcg_all, 0)
        ndcg_conditional = np.where(found, dcg / idcg_found, 0)
        hits = ranked_y[:, :k].sum(axis=1)
        recall = np.where(eligible, hits / np.maximum(truth,1), 0)
        result[f'ndcg@{k}'] = round(float(ndcg_all.mean()), 6)
        result[f'ndcg@{k}_eligible'] = round(float(ndcg_all[eligible].mean()), 6) if eligible.any() else np.nan
        result[f'ndcg@{k}_conditional'] = round(float(ndcg_conditional[found].mean()), 6) if found.any() else np.nan
        result[f'recall@{k}'] = round(float(recall.mean()), 6)
        result[f'recall@{k}_micro'] = round(float(hits.sum() / max(truth.sum(),1)), 6)
        result[f'hit@{k}'] = round(float((hits > 0).mean()), 6)
    result['roc_auc'] = round(float(roc_auc_score(y.ravel(), s.ravel())), 6)
    return result


def evaluate(split, name, scores, probabilities=False):
    cols = ['uid','item_id','group_id','retrieval_rank','label','future_positive_total']
    frame = pd.read_parquet(FEAT / f'features_{split}.parquet', columns=cols)
    result = ranking_metrics(frame, scores, name)
    result['split'] = split
    if probabilities:
        result['logloss'] = round(float(log_loss(frame.label, np.clip(scores, 1e-7, 1-1e-7))), 6)
    else:
        result['logloss'] = np.nan

    pd.DataFrame({'uid': frame.uid, 'item_id': frame.item_id,
                  'group_id': frame.group_id, 'label': frame.label,
                  'score': np.asarray(scores, dtype='float32')}).to_parquet(
        OUT / f'pred_{name}_{split}.parquet', index=False)
    del frame
    return result


## Retrieval

In [10]:
rows = []
for split in ('train','valid'):
    ranks = pd.read_parquet(FEAT / f'features_{split}.parquet', columns=['retrieval_rank'])
    rows.append(evaluate(split, 'R0', -ranks.retrieval_rank.to_numpy(), probabilities=False))
    del ranks
valid_results = pd.DataFrame(rows)
display(valid_results[['split','model','ndcg@20','ndcg@20_conditional',
                       'recall@20','roc_auc','positive_groups_pct']])


,split,model,ndcg@20,ndcg@20_conditional,recall@20,roc_auc,positive_groups_pct
0,train,R0,0.022323,0.073326,0.017523,0.667968,38.82
1,valid,R0,0.022924,0.073716,0.017861,0.663389,40.12


## CatBoost

In [15]:
def train_catboost(name, feature_names):
    started = time.perf_counter()
    train_cols = feature_names + ['label']
    train = pd.read_parquet(FEAT / 'features_train.parquet', columns=train_cols)
    valid = pd.read_parquet(FEAT / 'features_valid.parquet', columns=train_cols)
    print(name, 'train:', train.shape, 'valid:', valid.shape)

    xtr = train[feature_names].to_numpy(dtype=np.float32, copy=True)
    ytr = train.label.to_numpy(dtype=np.int8)
    xva = valid[feature_names].to_numpy(dtype=np.float32, copy=True)
    yva = valid.label.to_numpy(dtype=np.int8)
    del train, valid
    gc.collect()

    model = CatBoostClassifier(iterations=ITERATIONS, depth=DEPTH,
                               learning_rate=LEARNING_RATE, l2_leaf_reg=5,
                               loss_function='Logloss', eval_metric='Logloss',
                               task_type='CPU', devices='0', border_count=64,
                               random_seed=SEED, verbose=100)
    model.fit(xtr, ytr, eval_set=(xva, yva),
              early_stopping_rounds=100, use_best_model=True)
    preds = model.predict_proba(xva)[:, 1].astype('float32')
    model.save_model(str(MODELS / f'{name}.cbm'))
    importance = pd.DataFrame({'feature': feature_names,
                               'importance': model.get_feature_importance()})
    importance.sort_values('importance', ascending=False).to_csv(
        OUT / f'feature_importance_{name}.csv', index=False)
    elapsed = round(time.perf_counter() - started, 1)
    result = evaluate('valid', name, preds, probabilities=True)
    result['fit_predict_seconds'] = elapsed
    result['best_iteration'] = model.best_iteration_
    print(name, 'valid NDCG@20:', result['ndcg@20'],
          '| best iteration:', model.best_iteration_, '| seconds:', elapsed)
    del xtr, xva, ytr, yva, preds, model
    gc.collect()
    return result


In [16]:
for name, feature_names in FEATURE_SETS.items():
    valid_results = pd.concat([
        valid_results, pd.DataFrame([train_catboost(name, feature_names)])
    ], ignore_index=True)
    valid_results.to_csv(OUT / 'validation_metrics.csv', index=False)

display(valid_results[['model','ndcg@20','ndcg@20_conditional',
                       'recall@20','roc_auc','logloss','fit_predict_seconds']].round(5))


R1 train: (6000000, 43) valid: (6000000, 43)
0:	learn: 0.5600521	test: 0.5612511	best: 0.5612511 (0)	total: 718ms	remaining: 9m 33s
100:	learn: 0.0307845	test: 0.0323069	best: 0.0323069 (100)	total: 1m 3s	remaining: 7m 21s
200:	learn: 0.0299784	test: 0.0320071	best: 0.0320071 (200)	total: 2m 1s	remaining: 6m 3s
300:	learn: 0.0294500	test: 0.0318929	best: 0.0318929 (300)	total: 3m	remaining: 4m 58s
400:	learn: 0.0290599	test: 0.0318438	best: 0.0318438 (400)	total: 3m 58s	remaining: 3m 57s
500:	learn: 0.0287020	test: 0.0318469	best: 0.0318409 (446)	total: 4m 57s	remaining: 2m 57s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.03184092702
bestIteration = 446

Shrink model to first 447 iterations.
R1 valid NDCG@20: 0.036383 | best iteration: 446 | seconds: 330.3
R2 train: (6000000, 66) valid: (6000000, 66)
0:	learn: 0.5664939	test: 0.5655862	best: 0.5655862 (0)	total: 624ms	remaining: 8m 18s
100:	learn: 0.0268165	test: 0.0283005	best: 0.0283005 (100)	total: 1m 13s	rem

,model,ndcg@20,ndcg@20_conditional,recall@20,roc_auc,logloss,fit_predict_seconds
0,R0,0.02232,0.07333,0.01752,0.66797,NaN,NaN
1,R0,0.02292,0.07372,0.01786,0.66339,NaN,NaN
2,R1,0.03638,0.12228,0.02812,0.85317,0.03184,330.3
3,R2,0.08662,0.31414,0.06268,0.91207,0.02762,553.8


## Валидация

In [17]:
fi = pd.read_csv(OUT / 'feature_importance_R2.csv')
display(fi.head(20))
view = valid_results.set_index('model')
print('Δ NDCG@20 (R2 − R1), validation:',
      round(view.loc['R2','ndcg@20'] - view.loc['R1','ndcg@20'], 6))
print('Δ NDCG@20 (R2 − R0), validation:',
      round(view.loc['R2','ndcg@20'] - view.loc['R0','ndcg@20'], 6))


,feature,importance
0,u_last_age_days,20.041845
1,ui_last_listen_age_days,5.837581
2,itemknn_score,5.541427
3,ui_last_age_days,5.372792
4,u_organic_30d,4.608040
5,u_pos_7d,3.225186
6,u_repeat_ratio_30d,2.984683
7,u_length_30d,2.975809
8,u_positive_rate_7d,2.693212
9,u_n_30d,2.507032


Δ NDCG@20 (R2 − R1), validation: 0.050238
Δ NDCG@20 (R2 − R0), validation: model
R0    0.064298
R0    0.063697
Name: ndcg@20, dtype: float64


## Тест

In [18]:
test_results = []
ranks = pd.read_parquet(FEAT / 'features_test.parquet', columns=['retrieval_rank'])
test_results.append(evaluate('test', 'R0', -ranks.retrieval_rank.to_numpy(), False))
del ranks

for name, feature_names in FEATURE_SETS.items():
    model = CatBoostClassifier()
    model.load_model(str(MODELS / f'{name}.cbm'))
    xte = pd.read_parquet(FEAT / 'features_test.parquet', columns=feature_names)
    preds = model.predict_proba(xte.to_numpy(dtype=np.float32))[:, 1].astype('float32')
    test_results.append(evaluate('test', name, preds, True))
    del xte, preds, model
    gc.collect()

test_results = pd.DataFrame(test_results)
test_results.to_csv(OUT / 'test_metrics.csv', index=False)
display(test_results[['model','ndcg@20','ndcg@20_conditional','recall@20',
                      'recall@20_micro','roc_auc','logloss']].round(5))


,model,ndcg@20,ndcg@20_conditional,recall@20,recall@20_micro,roc_auc,logloss
0,R0,0.02314,0.07402,0.01764,0.02952,0.66946,NaN
1,R1,0.03813,0.12861,0.02953,0.04378,0.85793,0.03118
2,R2,0.08547,0.31259,0.06279,0.08850,0.91468,0.02718


## Артефакты